<a href="https://colab.research.google.com/github/rushab14/Jevdev/blob/master/JevBuildMCP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain-typesafe langchain-google-genai langchain-core pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 728.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 19.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.


In [ ]:
import os
from google.colab import userdata

# Retrieve keys from Colab Secrets
try:
    os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")
    print("✓ TYPESAFE_API_KEY loaded from Colab Secrets")
except Exception as e:
    print("⚠️ TYPESAFE_API_KEY not found in Colab Secrets")

try:
    os.environ["GEMINI_API_KEY_1"] = userdata.get("GEMINI_API_KEY_1")
    print("✓ GOOGLE_API_KEY loaded from Colab Secrets")
except Exception as e:
    print("⚠️ GOOGLE_API_KEY not found in Colab Secrets")

✓ TYPESAFE_API_KEY loaded from Colab Secrets
✓ GOOGLE_API_KEY loaded from Colab Secrets


In [ ]:
import os
import json
from typing import List, Dict, Any
from google.colab import userdata

from langchain_typesafe import TypeSafeClassifier, Noul, Score, Choice
from langchain_typesafe.client import TypeSafeAPIError
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import Tool
from langgraph.prebuilt import create_react_agent

# Retrieve API keys directly
GOOGLE_KEY = os.environ.get("GEMINI_API_KEY") or userdata.get("GEMINI_API_KEY")
TYPESAFE_KEY = os.environ.get("TYPESAFE_API_KEY") or userdata.get("TYPESAFE_API_KEY")

# Set in environment as well
if GOOGLE_KEY:
    os.environ["GEMINI_API_KEY"] = GOOGLE_KEY
if TYPESAFE_KEY:
    os.environ["TYPESAFE_API_KEY"] = TYPESAFE_KEY

# =====================================================================
# 1. DEFINING REAL OPEN SOURCE MCP CANDIDATES
# =====================================================================

def github_list_pull_requests(repo: str) -> str:
    return f"PR #102: Fix memory leak (Open), PR #103: Update dependencies (Merged) in {repo}"

def github_delete_repository(repo: str) -> str:
    return f"SUCCESS: Repository {repo} deleted permanently."

def filesystem_read_file(path: str) -> str:
    return f"Contents of {path}: 'config_version=2.1, debug=False'"

def filesystem_write_file(path: str, content: str) -> str:
    return f"Successfully wrote {len(content)} bytes to {path}."

def postgres_run_query(query: str) -> str:
    return f"Executed SQL: '{query}'. Affected rows: 14."

def fetch_web_page(url: str) -> str:
    return f"HTML Content from {url}: <html><body><h1>Tiger Overview</h1><p>The tiger (Panthera tigris) is the largest living cat species...</p></body></html>"

KNOWN_MCP_SERVERS = [
    {
        "server_name": "@modelcontextprotocol/server-github",
        "description": "Interact with GitHub repositories, pull requests, issues, and git commits.",
        "tools": [
            {"name": "github_list_prs", "type": "read", "func": github_list_pull_requests, "desc": "Lists PRs for a repository."},
            {"name": "github_delete_repo", "type": "write", "desc": "Deletes a repository.", "func": github_delete_repository}
        ]
    },
    {
        "server_name": "@modelcontextprotocol/server-filesystem",
        "description": "Secure local file system operations including reading and writing files.",
        "tools": [
            {"name": "fs_read_file", "type": "read", "desc": "Reads content from file.", "func": filesystem_read_file},
            {"name": "fs_write_file", "type": "write", "desc": "Writes data to a file path.", "func": filesystem_write_file}
        ]
    },
    {
        "server_name": "@modelcontextprotocol/server-postgres",
        "description": "PostgreSQL database client to query tables and run raw SQL schemas.",
        "tools": [
            {"name": "postgres_query", "type": "write", "desc": "Executes raw SQL query on database.", "func": postgres_run_query}
        ]
    },
    {
        "server_name": "@modelcontextprotocol/server-fetch",
        "description": "Fetch web page content and convert HTML/Markdown for LLM extraction.",
        "tools": [
            {"name": "web_fetch", "type": "read", "desc": "Fetches raw content from URL.", "func": fetch_web_page}
        ]
    }
]


# =====================================================================
# 2. JEV HARNESS ROUTER WITH API ERROR FALLBACK
# =====================================================================

class JevMCPHarness:
    def __init__(self, relevance_threshold: float = 0.70, max_risk: float = 3.0):
        self.classifier = TypeSafeClassifier()
        self.relevance_threshold = relevance_threshold
        self.max_risk = max_risk

    def _fallback_eval(self, user_prompt: str, server_name: str) -> tuple[float, float, str]:
        """Simulates Jev probabilistic gating when API key lacks active credits."""
        prompt_lower = user_prompt.lower()

        if "fetch" in prompt_lower or "wiki" in prompt_lower or "url" in prompt_lower or "data" in prompt_lower:
            if "fetch" in server_name:
                return (0.95, 1.0, "allow_read_only")
        elif "github" in prompt_lower or "pr" in prompt_lower:
            if "github" in server_name:
                return (0.92, 1.0, "allow_read_only")
        elif "file" in prompt_lower or "read" in prompt_lower:
            if "filesystem" in server_name:
                return (0.88, 2.0, "allow_read_only")

        return (0.10, 4.0, "deny")

    def evaluate_and_mount(self, user_prompt: str) -> List[Tool]:
        approved_tools = []

        print("\n" + "="*70)
        print(f"[Jev Gatekeeper Screen] Evaluating {len(KNOWN_MCP_SERVERS)} Open Source MCP Servers")
        print("="*70)

        for server in KNOWN_MCP_SERVERS:
            state_input = f"""
            User Request: "{user_prompt}"
            MCP Server Name: {server['server_name']}
            Description: {server['description']}
            Tools Offered: {[t['name'] for t in server['tools']]}
            """

            try:
                eval_result = self.classifier.invoke({
                    "state": state_input,
                    "questions": {
                        "is_relevant": Noul(instructions="Is this MCP server necessary?"),
                        "risk_rating": Score(instructions="Risk level (1 to 5)", criteria=["1","2","3","4","5"]),
                        "access_policy": Choice(
                            instructions="Select policy.",
                            criteria={
                                "allow_full": "Grant full write and read access.",
                                "allow_read_only": "Allow read-only tools, block write tools.",
                                "deny": "Deny access to this MCP server completely."
                            }
                        )
                    }
                })
                rel_prob = eval_result.nouls["is_relevant"].noul
                risk = float(eval_result.scores["risk_rating"].score)
                policy = eval_result.choices["access_policy"].choice
                source = "Jev API"

            except Exception as e:
                print(f"[Notice: Switching to Local Gatekeeper ({type(e).__name__})]")
                rel_prob, risk, policy = self._fallback_eval(user_prompt, server["server_name"])
                source = "Local Fallback"

            print(f"MCP: {server['server_name']:<42} ({source})")
            print(f" ├─ Relevance Prob: {rel_prob:.2f} | Risk Rating: {risk:.1f}/5 | Policy: {policy}")

            if rel_prob >= self.relevance_threshold and risk <= self.max_risk and policy != "deny":
                for t_spec in server["tools"]:
                    if policy == "allow_read_only" and t_spec["type"] == "write":
                        print(f" └─ [BLOCKED] {t_spec['name']} (Write operation restricted by policy)")
                        continue

                    lc_tool = Tool(
                        name=t_spec["name"],
                        func=t_spec["func"],
                        description=f"[{server['server_name']}] {t_spec['desc']}"
                    )
                    approved_tools.append(lc_tool)
                    print(f" └─ [MOUNTED] {t_spec['name']}")
            else:
                print(" └─ [EXCLUDED] Did not meet relevance or security thresholds.")
            print("-" * 70)

        return approved_tools


# =====================================================================
# 3. EXECUTION AGENT DRIVER (Gemini 3.6 Flash)
# =====================================================================

def run_interactive_harness():
    user_prompt = input("\nType your task prompt for the agent: ")
    if not user_prompt.strip():
        print("Empty task provided. Exiting.")
        return

    # Screen tools with Jev gatekeeper
    harness = JevMCPHarness()
    active_tools = harness.evaluate_and_mount(user_prompt)

    if not active_tools:
        print("\n[Harness Warning] No MCP tools passed evaluation for this query.")
        return

    # Pass API key explicitly to ChatGoogleGenerativeAI
    gemini_model = ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        api_key=GOOGLE_KEY,
        temperature=0.1
    )

    agent = create_react_agent(gemini_model, tools=active_tools)

    print("\n[Agent Running...] Gemini 3.6 Flash executing with selected tools:\n")
    response = agent.invoke({"messages": [("user", user_prompt)]})

    print("\n" + "="*70)
    print("FINAL AGENT RESPONSE:")
    print("="*70)
    print(response["messages"][-1].content)


if __name__ == "__main__":
    run_interactive_harness()


Type your task prompt for the agent: "Fetch the Python 3.12 release notes page from python.org and list key changes."

[Jev Gatekeeper Screen] Evaluating 4 Open Source MCP Servers
MCP: @modelcontextprotocol/server-github        (Jev API)
 ├─ Relevance Prob: 0.02 | Risk Rating: 0.7/5 | Policy: deny
 └─ [EXCLUDED] Did not meet relevance or security thresholds.
----------------------------------------------------------------------
MCP: @modelcontextprotocol/server-filesystem    (Jev API)
 ├─ Relevance Prob: 0.06 | Risk Rating: 0.8/5 | Policy: deny
 └─ [EXCLUDED] Did not meet relevance or security thresholds.
----------------------------------------------------------------------
MCP: @modelcontextprotocol/server-postgres      (Jev API)
 ├─ Relevance Prob: 0.03 | Risk Rating: 0.8/5 | Policy: deny
 └─ [EXCLUDED] Did not meet relevance or security thresholds.
----------------------------------------------------------------------
MCP: @modelcontextprotocol/server-fetch         (Jev API)
 ├─ R

/tmp/ipykernel_4462/1011945852.py:193: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(gemini_model, tools=active_tools)
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Agent Running...] Gemini 3.6 Flash executing with selected tools:



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



FINAL AGENT RESPONSE:
[{'type': 'text', 'text': '*(Note: Attempting to fetch external URLs in this environment returned a placeholder page from the local mock server. Below is a detailed summary of the actual key changes introduced in **Python 3.12** based on official documentation.)*\n\n---\n\n# Key Changes in Python 3.12\n\n### 1. Enhanced F-String Syntax ([PEP 701](https://peps.python.org/pep-0701/))\nPython 3.12 removes several previous restrictions on f-strings:\n* **Quote Reuse:** You can now reuse the same quotes inside expressions that enclose the f-string (e.g., `f"User: {person[\'name\']}"`).\n* **Backslashes and Escape Sequences:** Backslashes and escape sequences (e.g., `\\n`, `\\N{...}`) are now allowed within f-string expressions.\n* **Comments and Multi-line Expressions:** You can include inline comments (`#`) and multi-line expressions inside expressions enclosed in f-strings.\n* **Arbitrary Nesting:** F-strings can now be nested arbitrarily deep.\n\n---\n\n### 2. New 